Exploración inicial
Importamos pandas con un alias, y le decimos que lea el dataset. Le decimos con shape que nos digas cuántas filas y columnas tiene.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/jobs_in_data.csv')
df.shape

(9355, 12)

Ahora mostramos las primeras 5 filas del dataset. (Con df.tail (5) veríamos las últimas).

In [2]:
df.head()

,work_year,job_title,job_category,salary_currency,salary,salary_in_usd,employee_residence,experience_level,employment_type,work_setting,company_location,company_size
0,2023,Data DevOps Engineer,Data Engineering,EUR,88000,95012,Germany,Mid-level,Full-time,Hybrid,Germany,L
1,2023,Data Architect,Data Architecture and Modeling,USD,186000,186000,United States,Senior,Full-time,In-person,United States,M
2,2023,Data Architect,Data Architecture and Modeling,USD,81800,81800,United States,Senior,Full-time,In-person,United States,M
3,2023,Data Scientist,Data Science and Research,USD,212000,212000,United States,Senior,Full-time,In-person,United States,M
4,2023,Data Scientist,Data Science and Research,USD,93300,93300,United States,Senior,Full-time,In-person,United States,M


Mostramos tipos de datos 

In [3]:
df.dtypes

work_year             int64
job_title               str
job_category            str
salary_currency         str
salary                int64
salary_in_usd         int64
employee_residence      str
experience_level        str
employment_type         str
work_setting            str
company_location        str
company_size            str
dtype: object

Estadísticas básicas. Aquí nos da estadísticas para cada columna numérica: count (cuántos valores no nulos hay), mean (media aritmética), std (desviación estándar), min (Valor mínimo), 25% (Percentil 25, el 25% de los datos está por debajo de este valor), mediana (valor central del dataset), Percentil 75 (el 75% de los datos está por debajo de este valor), max (valor máximo).

In [4]:
df.describe()

,work_year,salary,salary_in_usd
count,9355.000000,9355.000000,9355.000000
mean,2022.760449,149927.981293,150299.495564
std,0.519470,63608.835387,63177.372024
min,2020.000000,14000.000000,15000.000000
25%,2023.000000,105200.000000,105700.000000
50%,2023.000000,143860.000000,143000.000000
75%,2023.000000,187000.000000,186723.000000
max,2023.000000,450000.000000,450000.000000


Lectura de resultados actual: El dataset cubre 2020 a 2023 (min/max de work_year). El salario medio es $150.299 — bastante alto. El mínimo es $15.000 y el máximo $450.000 — hay una dispersión enorme. El 75% de los registros está por debajo de $186.723, pero el máximo llega a $450k — esos valores extremos son outliers.

Ahora necesitamos saber si hay columnas con datos que faltan.

In [6]:
df.isnull().sum()

work_year             0
job_title             0
job_category          0
salary_currency       0
salary                0
salary_in_usd         0
employee_residence    0
experience_level      0
employment_type       0
work_setting          0
company_location      0
company_size          0
dtype: int64

El dataset está completo, sin ningún valor nulo.

Ahora veamos los valores únicos de las columnas más importantes. Esto nos dice exáctamente qué valores distintos existen en cada columna de texto. Importante para saber si hay variantes raras o errores tipográficos que limpiar.

In [7]:
print(df['experience_level'].unique())
print(df['work_setting'].unique())
print(df['employment_type'].unique())
print(df['company_size'].unique())

<ArrowStringArray>
['Mid-level', 'Senior', 'Executive', 'Entry-level']
Length: 4, dtype: str
<ArrowStringArray>
['Hybrid', 'In-person', 'Remote']
Length: 3, dtype: str
<ArrowStringArray>
['Full-time', 'Part-time', 'Contract', 'Freelance']
Length: 4, dtype: str
<ArrowStringArray>
['L', 'M', 'S']
Length: 3, dtype: str


Todo limpio y consistente.

Limpieza de datos: Importamos las funciones de transform.py y aplicamos el pipeline de limpieza sobre el dataset raw.
Las transformaciones aplicadas son: estandarización de nombres de columna, eliminación de duplicados, y normalización de columnas de texto.

In [3]:
import sys
sys.path.append('../src')

from transform import load_raw, clean, save_processed

# Cargamos el raw
df_raw = load_raw('../data/raw/jobs_in_data.csv')

# Limpiamos
df_clean = clean(df_raw)

# Comprobamos que todo está bien
print(f'Filas antes: {len(df_raw)}')
print(f'Filas después: {len(df_clean)}')
print(f'Columnas: {list(df_clean.columns)}')
print(f'work_setting únicos: {df_clean["work_setting"].unique()}')
print(f'experience_level únicos: {df_clean["experience_level"].unique()}')

Filas antes: 9355
Filas después: 5341
Columnas: ['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']
work_setting únicos: <ArrowStringArray>
['Hybrid', 'In-Person', 'Remote']
Length: 3, dtype: str
experience_level únicos: <ArrowStringArray>
['Mid-level', 'Senior', 'Executive', 'Entry-level']
Length: 4, dtype: str


Observaciones: todas las columnas con guiones bajo y en minúsculas, work_setting normalizado: Hybrid, In-Person, Remote. experience_level normalizado y legible.
Hemos perdido 4014 filas. Investiguemos por qué tantas.

In [2]:
# Cuántos duplicados exactos hay
print(f'Duplicados exactos: {df_raw.duplicated().sum()}')

# Ver un ejemplo de fila "duplicada"
print(df_raw[df_raw.duplicated(keep=False)].head(10))

Duplicados exactos: 4014
    work_year                  job_title                    job_category  \
1        2023             Data Architect  Data Architecture and Modeling   
2        2023             Data Architect  Data Architecture and Modeling   
3        2023             Data Scientist       Data Science and Research   
4        2023             Data Scientist       Data Science and Research   
5        2023             Data Scientist       Data Science and Research   
6        2023             Data Scientist       Data Science and Research   
16       2023               Data Analyst                   Data Analysis   
17       2023             Data Scientist       Data Science and Research   
18       2023             Data Scientist       Data Science and Research   
21       2023  Machine Learning Engineer         Machine Learning and AI   

   salary_currency  salary  salary_in_usd employee_residence experience_level  \
1              USD  186000         186000      United Sta

Las filas 3, 4, 5, 6 — mismo año, mismo título, mismo salario, mismo país, mismo nivel... son registros idénticos en todas las columnas.
En un dataset de ofertas de trabajo esto es normal — varias personas con el mismo perfil y salario reportaron su empleo. No son errores, son datos reales repetidos.
Para un análisis de mercado laboral, los mantendremos los datos. 
Si hay 10 Data Scientists con el mismo perfil, eso nos dice que ese rol es frecuente — eliminarlos distorsionaría las estadísticas de demanda.

In [1]:
import sys
sys.path.append('../src')

from transform import load_raw, clean, save_processed

# Cargamos el raw
df_raw = load_raw('../data/raw/jobs_in_data.csv')

# Limpiamos
df_clean = clean(df_raw)

# Comprobamos que todo está bien
print(f'Filas antes: {len(df_raw)}')
print(f'Filas después: {len(df_clean)}')
print(f'Columnas: {list(df_clean.columns)}')
print(f'work_setting únicos: {df_clean["work_setting"].unique()}')
print(f'experience_level únicos: {df_clean["experience_level"].unique()}')

Filas antes: 9355
Filas después: 9355
Columnas: ['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']
work_setting únicos: <ArrowStringArray>
['Hybrid', 'In-Person', 'Remote']
Length: 3, dtype: str
experience_level únicos: <ArrowStringArray>
['Mid-level', 'Senior', 'Executive', 'Entry-level']
Length: 4, dtype: str


Ahora guardamos el CSV limpio

In [2]:
save_processed(df_clean, '../data/processed/jobs_clean.csv')

Guardado: 9355 filas en ../data/processed/jobs_clean.csv


Feature engineering: creamos nuevas columnas a partir de datos que ya tenemos, para que el análisis sea más rico y más fácil de visualizar después

salary_range = El salario es un número, lo hemos convertido en una categoría legible, eso permite agrupar y comparar
experience_group = Tenía los niveles de experiencia en inglés formal, lo he mapeado a etiquetas más limpias
is_remote = Convierte texto en un booleano, así es más fácil de filtrar
year_group = toma valores numéricos o de fecha, define unos límites y asigna una eqtiqueta a cada valor segun en qué tramo cae.

In [ ]:
import sys
sys.path.append('..')

from src.transform import load_raw, clean, engineer_features

df = load_raw('data/raw/jobs_in_data.csv')
df = clean(df)
df = engineer_features(df)

# Comprobamos que las columnas nuevas están
print(df[['salary_in_usd', 'salary_range', 'experience_level', 'experience_group', 'work_setting', 'is_remote', 'work_year', 'year_group']].head(10))

ModuleNotFoundError: No module named 'src'